# Image Colorization (Local Model)

This notebook trains a colorization model that predicts color (AB channels)
from grayscale input (L channel).

The model is optimized to:
- Produce visible colors
- Avoid grayscale outputs
- Train quickly on limited hardware

In [1]:
!pip install tensorflow tensorflow-datasets opencv-python

In [2]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import cv2

from tensorflow.keras.layers import Conv2D, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [3]:
dataset = tfds.load("oxford_iiit_pet", split="train[:60%]")

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/oxford_iiit_pet/incomplete.3VSUC8_4.0.0/oxford_iiit_pet-train.tfrecord*...…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/oxford_iiit_pet/incomplete.3VSUC8_4.0.0/oxford_iiit_pet-test.tfrecord*...:…

Dataset oxford_iiit_pet downloaded and prepared to /root/tensorflow_datasets/oxford_iiit_pet/4.0.0. Subsequent calls will reuse this data.


In [4]:
IMG_SIZE = 128

def preprocess(sample):
    img = sample["image"]
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) / 255.0

    def to_lab(x):
        x = (x * 255).numpy().astype("uint8")
        lab = cv2.cvtColor(x, cv2.COLOR_RGB2LAB)

        L = lab[:, :, 0] / 255.0

        AB = lab[:, :, 1:] / 128.0 - 1.0

        return L[..., None], AB

    L, AB = tf.py_function(to_lab, [img], [tf.float32, tf.float32])

    L.set_shape((IMG_SIZE, IMG_SIZE, 1))
    AB.set_shape((IMG_SIZE, IMG_SIZE, 2))

    return L, AB

train_ds = dataset.map(preprocess).batch(32).prefetch(tf.data.AUTOTUNE)

In [5]:
def build_model():
    inp = Input((IMG_SIZE, IMG_SIZE, 1))

    x = Conv2D(64, 3, padding="same", activation="relu")(inp)
    x = Conv2D(128, 3, padding="same", activation="relu")(x)
    x = Conv2D(256, 3, padding="same", activation="relu")(x)

    x = Conv2D(128, 3, padding="same", activation="relu")(x)
    x = Conv2D(64, 3, padding="same", activation="relu")(x)

    out = Conv2D(2, 1, activation="tanh")(x)

    return Model(inp, out)

model = build_model()
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 128, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 64)   │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 128, 128, 256)  │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 128, 128, 128)  │       295,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 128, 128, 64)   │        73,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 128, 128, 2)    │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 738,626 (2.82 MB)

 Trainable params: 738,626 (2.82 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
model.compile(
    optimizer=Adam(1e-3),
    loss="mae"
)

In [7]:
model.fit(
    train_ds,
    epochs=10
)

Epoch 1/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 74s 471ms/step - loss: 0.0681
Epoch 2/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 35s 511ms/step - loss: 0.0650
Epoch 3/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 38s 549ms/step - loss: 0.0647
Epoch 4/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 39s 563ms/step - loss: 0.0645
Epoch 5/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 38s 554ms/step - loss: 0.0645
Epoch 6/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 39s 563ms/step - loss: 0.0644
Epoch 7/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 39s 564ms/step - loss: 0.0644
Epoch 8/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 39s 561ms/step - loss: 0.0644
Epoch 9/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 38s 551ms/step - loss: 0.0643
Epoch 10/10
69/69 ━━━━━━━━━━━━━━━━━━━━ 38s 555ms/step - loss: 0.0643


In [8]:
model.save("colorization_model.keras")

In [9]:
from google.colab import files
files.download("colorization_model.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>